# Part One
## Here we will do these things - 
 - #### Process Text
 - #### Clean Text
 - #### Tokenize the text and create sequences with keras 

RNN forgets the information trained which was trained earlier - after a while of training new and new data. 

Long short term memory (LSTM) cellw as created to help address these RNN issues. 


What does RNN do? 

It gives input - gets output - and gives this output as input to same network - this is time series data - 
so it checkes input for time instances 

In [1]:
def read_file(filepath):
    with open(filepath) as f:
        str_text = f.read()

    return str_text

In [21]:
# read_file("L:\\Workspace\\Resources\\UPDATED_NLP_COURSE\\06-Deep-Learning\\moby_dick_four_chapters.txt")

In [3]:
import spacy

In [6]:
nlp = spacy.load('en_core_web_lg', disable=['parser','tagger','ner'])

In [7]:
nlp.max_length = 1198623

In [8]:
# I wanna remove punctuations, because they occur often and we dont want out model to get into training
def sep_punc(doc_text):
    return [token.text.lower() for token in nlp(doc_text) if token.text not in '\n\n \n\n\n!"-#$%&()--.*+,-/:;<=>?@[\\]^_`{|}~\t\n ']

In [9]:
d = read_file("L:/Workspace/Resources/UPDATED_NLP_COURSE/06-Deep-Learning/moby_dick_four_chapters.txt")

In [10]:
tokens = sep_punc(d)

C:\Users\Asus\anaconda3\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [69]:
len(tokens)

11338

In [23]:
tokens[:30]

['call',
 'me',
 'ishmael',
 'some',
 'years',
 'ago',
 'never',
 'mind',
 'how',
 'long',
 'precisely',
 'having',
 'little',
 'or',
 'no',
 'money',
 'in',
 'my',
 'purse',
 'and',
 'nothing',
 'particular',
 'to',
 'interest',
 'me',
 'on',
 'shore',
 'i',
 'thought',
 'i']

In [13]:
# We will pass first 25 words of sentences, and we will let our model predict 26th. 

In [14]:
train_len = 25 +1 
text_seq = []
for i in range(train_len, len(tokens)):
    seq = tokens[i-train_len:i]
    text_seq.append(seq)

In [15]:
' '.join(text_seq[0])

'call me ishmael some years ago never mind how long precisely having little or no money in my purse and nothing particular to interest me on'

In [16]:
' '.join(text_seq[2])

'ishmael some years ago never mind how long precisely having little or no money in my purse and nothing particular to interest me on shore i'

# What did we do here? 
We made nlp object. Then we removed parser, tagger, and ner (named entity recognizer) from it. 
Then we made a tokens through the text file - how? By removing the puncutations and new lines - because we dont wanna make them our training. 
Then now - we took 25 words, and we ask what is next word? Its nto doing any work, just checking in the list of tokens, and giving us another word. That's all. 

In [19]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [24]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(text_seq)

In [25]:
sequences = tokenizer.texts_to_sequences(text_seq)

In [32]:
# tokenizer.index_word

We simply gave tokenizer the text sequences we created - and then we got tokens - id for each word 

In [70]:
vocab_size = len(tokenizer.word_counts) # Tells us how many times a word comes 
vocab_size

2718

In [ ]:
len(tokenizer.word_counts)

In [33]:
len(sequences)

11312

In [31]:
import numpy as np
sequences = np.array(sequences)
sequences

array([[ 956,   14,  263, ..., 2713,   14,   24],
       [  14,  263,   51, ...,   14,   24,  957],
       [ 263,   51,  261, ...,   24,  957,    5],
       ...,
       [ 952,   12,  166, ...,  262,   53,    2],
       [  12,  166, 2712, ...,   53,    2, 2718],
       [ 166, 2712,    3, ...,    2, 2718,   26]])

# Part Two - 
- #### In this part we will create LSTM.
- #### Split the data.
- #### And fit our model to the data 

In [34]:
from keras.utils import to_categorical

Now how to get our X and y? 
So for X we need every item of our OUTER array - and every time except last one on our inner array -  

That means - 
sequences[: , :-1] - meaning take all the sequences - and in all those take everything except last item 

In [49]:
X = sequences[:, :-1]

Now we take y - we take all the sequences from outer array . 
- sequences[:, :] - 
but in this we take only last character - or last item for our y 
- sequences(:,-1]

In [40]:
y = sequences[:,-1]

In [74]:
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [46]:
y = to_categorical(y, num_classes=vocab_size+1)

In [50]:
seq_len = X.shape[1]

In [52]:
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding

In [61]:
def create_model(vocab_size, seq_len):
    model = Sequential()
    model.add(Embedding(vocab_size, seq_len, input_length=seq_len))
    # we are giving input_dimension - vocab_size , and output_dimension = seq_len, and input_len 
    model.add(LSTM((seq_len*3), return_sequences=True))
    # We added 3 times out sequence len 
    model.add(LSTM(50))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(vocab_size, activation='softmax'))

    model.compile(loss = 'categorical_crossentropy',optimizer='adam',metrics = ['accuracy'])

    model.summary()
    return model

In [62]:
model = create_model(vocab_size+1, seq_len)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [77]:
model.fit(X, y, batch_size=128, epochs=300,verbose=1)

Epoch 1/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 4s 46ms/step - accuracy: 0.6320 - loss: 1.4616
Epoch 2/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.6394 - loss: 1.4421
Epoch 3/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.6351 - loss: 1.4401
Epoch 4/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.6392 - loss: 1.4385
Epoch 5/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.6444 - loss: 1.4290
Epoch 6/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.6488 - loss: 1.4005
Epoch 7/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.6530 - loss: 1.3833
Epoch 8/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.6582 - loss: 1.3764
Epoch 9/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.6578 - loss: 1.3610
Epoch 10/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 6s 47ms/step - accuracy: 0.6624 - loss: 1.3543
Epoch 11/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.6638 - loss: 1.3508
Epoch 12/300
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step

In [78]:
from pickle import dump,load
model.save("L:\\Workspace\\Models\\mobidick_epoch300.h5")

We will run it on 1 or 2 epochs for now - but we should train it on atleast 200 epochs so that it learns something better, and then we can save it.


In [101]:
dump(tokenizer, open('L:\\Workspace\\Models\\my_simple_tokenizer','wb'))

# Part THREE 
- #### Generating Text 

In [79]:
from keras.preprocessing.sequence import pad_sequences

In [96]:

def generate_text(model, tokenizer, seq_len, seed_text, num_gen_words):
    output_text = []

    input_text = seed_text
    #  Seed text is out input - 25 words sentece. We will get this - and generate a new word. 
    # And then we will chop off the very first word and take new generated word in the end to get the new sequence of 25 words- aka new seed text 
    for i in range(num_gen_words):
        # I will run this - how many words we wnna generate - 1
        # and then we we transform raw text data to seq of numbers
        encoded_text = tokenizer.texts_to_sequences([input_text])[0]

        # If seed text is too short of long, we need to pad it - to cut or add on it. 
        pad_encoding = pad_sequences([encoded_text], maxlen = seq_len, truncating='pre')

        # I will predict class probabilities for each word - which word would come next - that probrabilities. 
        # We will do argmax on each one to get the index of word which had most probability of coming next. 
        preds = model.predict(pad_encoding, verbose=0)[0]
        predicted_word_ind = np.argmax(preds)

        # Now we got the word using index. 
        pred_word = tokenizer.index_word[predicted_word_ind]

        # Now I will add a space and add predicted word in the input. And now I can later trim it from starting - in the starting of the loop. 
        input_text += ' ' + pred_word

        # After a lot of time - we will be generating on only predited words. 
        output_text.append(pred_word)
    
    return ' '.join(output_text)

In [90]:
text_seq[0]

['call',
 'me',
 'ishmael',
 'some',
 'years',
 'ago',
 'never',
 'mind',
 'how',
 'long',
 'precisely',
 'having',
 'little',
 'or',
 'no',
 'money',
 'in',
 'my',
 'purse',
 'and',
 'nothing',
 'particular',
 'to',
 'interest',
 'me',
 'on']

In [107]:
import random
random.seed(101)
random_pick = random.randint(0,len(text_seq))

In [108]:
random_seed_text = text_seq[random_pick]

In [109]:
# random_seed_text

In [116]:
# seed_text = ' '.join(random_seed_text)
# seed_text = "I thought I would sail about a little and see the watery part of the world It is a way I have of driving off"
seed_text = "I walked along the silent harbor at dawn watching the restless water while thoughts of distant voyages and uncharted horizons stirred quietly within my mind"
# the spleen and regulating the circulation.'
seed_text

'I walked along the silent harbor at dawn watching the restless water while thoughts of distant voyages and uncharted horizons stirred quietly within my mind'

In [115]:
generate_text(model, tokenizer, seq_len, seed_text=seed_text, num_gen_words=50)

"the spleen and regulating the circulation whenever i find myself growing grim about the mouth whenever you put the second chuckle and boots about now that lazarus more n't see it previous to by current of the picture at all if over the good pieces of the tall yellow manner"

In [117]:
generate_text(model, tokenizer, seq_len, seed_text=seed_text, num_gen_words=50)

"male and being so aloof and thing what 's checkered that a bag and found yourself with it look was this harpooneer else for my sort of systematic something in a pulpit bed about sabbee the streets and whittling me and a black window or think the story that it"

In [98]:
text_seq

[['call',
  'me',
  'ishmael',
  'some',
  'years',
  'ago',
  'never',
  'mind',
  'how',
  'long',
  'precisely',
  'having',
  'little',
  'or',
  'no',
  'money',
  'in',
  'my',
  'purse',
  'and',
  'nothing',
  'particular',
  'to',
  'interest',
  'me',
  'on'],
 ['me',
  'ishmael',
  'some',
  'years',
  'ago',
  'never',
  'mind',
  'how',
  'long',
  'precisely',
  'having',
  'little',
  'or',
  'no',
  'money',
  'in',
  'my',
  'purse',
  'and',
  'nothing',
  'particular',
  'to',
  'interest',
  'me',
  'on',
  'shore'],
 ['ishmael',
  'some',
  'years',
  'ago',
  'never',
  'mind',
  'how',
  'long',
  'precisely',
  'having',
  'little',
  'or',
  'no',
  'money',
  'in',
  'my',
  'purse',
  'and',
  'nothing',
  'particular',
  'to',
  'interest',
  'me',
  'on',
  'shore',
  'i'],
 ['some',
  'years',
  'ago',
  'never',
  'mind',
  'how',
  'long',
  'precisely',
  'having',
  'little',
  'or',
  'no',
  'money',
  'in',
  'my',
  'purse',
  'and',
  'nothing',
 